In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import pandas as pd
import os
import shutil
from numba import njit
from google.colab import files

# ==========================================
# 1. パラメータ設定
# ==========================================
N_LIST = [1000, 10000, 100000] # スケールごとの比較 (計算時間を考慮し10万まで)
BURN_IN_STEPS = 100
ATTACK_STEPS = 100
TOTAL_STEPS = BURN_IN_STEPS + ATTACK_STEPS
DT = 0.1

THETA_DETECT = 0.18 # センサー閾値
LAMBDA_DUMMY = 0.5  # 毎ステップ追加されるダミーデータの割合 (ノード数に対する比率)
B_PAYLOAD = 0.95    # 攻撃ペイオフ

# 保存用ディレクトリの作成
OUTPUT_DIR = "simulation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. Numbaによる高速SDE積分エンジン
# ==========================================
@njit
def sde_update(X, D, degrees, b_val, N, dt=0.1, sigma=0.05, gamma=6.0, k_break=0.6, k_restore=0.1):
    # 平均場 P の近似計算 (スケールフリー性を考慮した重み付き平均)
    total_degree = np.sum(degrees)
    mean_field = np.sum(X * degrees) / total_degree if total_degree > 0 else 0.0

    # 確率的ノイズ
    noise_X = np.random.normal(0, 1, N) * np.sqrt(dt)

    # dX: Conformityの更新
    dX = (mean_field + b_val - X) * dt + sigma * noise_X
    X_new = np.minimum(np.maximum(X + dX, 0.0), 1.0)

    # dD: トポロジカル欠陥の更新
    repair_capacity = np.exp(-gamma * D)
    dD = (k_break * X_new * (1.0 - D) - k_restore * (1.0 - X_new) * D * repair_capacity) * dt
    D_new = np.minimum(np.maximum(D + dD, 0.0), 1.0)

    return X_new, D_new

# ==========================================
# 3. メインシミュレーションループ
# ==========================================
ks_results = []
sensor_results = []

for N in N_LIST:
    print(f"--- Simulating N = {N} ---")
    # 初期化: 一様分布 U(0, 0.2) で初期値依存性を排除
    X = np.random.uniform(0.0, 0.2, N)
    D = np.random.uniform(0.0, 0.1, N)

    # 次数分布の生成 (べき乗則近似)
    degrees = np.random.pareto(2.5, N) + 1.0

    # ハブノードの特定
    hub_idx = np.argmax(degrees)
    b_val = np.zeros(N)

    # KS統計量記録用
    X_history = []

    # --- Phase 0: Burn-in (自己組織化の証明) ---
    for step in range(BURN_IN_STEPS):
        X, D = sde_update(X, D, degrees, b_val, N, dt=DT)

        if step % 10 == 0:
            X_history.append(X.copy())
            if len(X_history) > 1:
                # 10ステップ前との分布の差 (KS統計量) を計算
                stat, _ = ks_2samp(X_history[-1], X_history[-2])
                ks_results.append({"N": N, "Step": step, "KS_Statistic": stat})

    # --- Phase 1 & 2: 攻撃とダミーデータによるセンサー希釈 ---
    total_dummies = 0
    b_val[hub_idx] = B_PAYLOAD # ハブへの偽ペイオフ注入

    for step in range(BURN_IN_STEPS, TOTAL_STEPS):
        X, D = sde_update(X, D, degrees, b_val, N, dt=DT)

        # ダミーデータの生成 (ノイズフラッディング)
        # 毎ステップ LAMBDA_DUMMY の割合で無害なデータ(X=0相当)が増殖すると仮定
        added_dummies = int(N * LAMBDA_DUMMY * DT)
        total_dummies += added_dummies

        # マクロセンサー X_A の計算 (ダミーによる統計的希釈)
        true_sum_X = np.sum(X)
        sensor_X_A = true_sum_X / (N + total_dummies)

        # 実態の欠陥率 (Systemic Defect)
        mean_D = np.mean(D)

        sensor_results.append({
            "N": N,
            "Step": step,
            "Sensor_X_A": sensor_X_A,
            "Actual_Defect_D": mean_D,
            "Dummy_Count": total_dummies
        })

# ==========================================
# 4. データ保存と可視化 (CSV & グラフ)
# ==========================================
# データフレーム化
df_ks = pd.DataFrame(ks_results)
df_sensor = pd.DataFrame(sensor_results)

df_ks.to_csv(f"{OUTPUT_DIR}/ks_convergence.csv", index=False)
df_sensor.to_csv(f"{OUTPUT_DIR}/sensor_dynamics_and_defect.csv", index=False)

# グラフ1: KS統計量の収束 (キャリブレーション不要の証明)
plt.figure(figsize=(10, 6))
for N in N_LIST:
    subset = df_ks[df_ks["N"] == N]
    plt.plot(subset["Step"], subset["KS_Statistic"], marker='o', label=f"N={N}")
plt.title("Ergodic Self-Organization: KS Statistic Convergence")
plt.xlabel("Burn-in Steps")
plt.ylabel("KS Statistic (Distribution Drift)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig1_KS_Convergence.png", dpi=300)
plt.close()

# グラフ2: センサー盲目化とトポロジー崩壊 (大スケール N=100,000)
plt.figure(figsize=(10, 6))
subset_sensor = df_sensor[df_sensor["N"] == N_LIST[-1]]
plt.plot(subset_sensor["Step"], subset_sensor["Sensor_X_A"], color='blue', label="Macro Sensor $X_A$ (Diluted)")
plt.plot(subset_sensor["Step"], subset_sensor["Actual_Defect_D"], color='red', linestyle='--', label="Actual Topological Defect $\\beta_1$")
plt.axhline(y=THETA_DETECT, color='black', linestyle=':', label="Detection Threshold $\\theta=0.18$")
plt.title(f"Thermodynamic Shielding & Systemic Cascade (N={N_LIST[-1]})")
plt.xlabel("Time Step (Post Burn-in)")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig2_Sensor_Failure_and_Cascade.png", dpi=300)
plt.close()

# ==========================================
# 5. ZIP圧縮と自動ダウンロード
# ==========================================
shutil.make_archive("Simulation_Results_Hypothesis2", 'zip', OUTPUT_DIR)
print("シミュレーション完了。結果をZIP圧縮してダウンロードします...")
files.download("Simulation_Results_Hypothesis2.zip")

--- Simulating N = 1000 ---
--- Simulating N = 10000 ---
--- Simulating N = 100000 ---
シミュレーション完了。結果をZIP圧縮してダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>